# 第16回　ANN と CNN・モデル比較
***
> **前提**: 第15回 SimpleMLP の続きです。第7回の線形回帰に相当する **LinearNet**（隠れ層なし），多層 **DeepMLP**，**SimpleCNN** の3モデルを同条件で学習し精度を比較します。第12回で学んだ混同行列も使います。

> ⚠️ **この課題で身につけること：ネットワークを一から書く力ではなく「AI（ニューラルネット）の中身の理解」です。**
>
> モデルのコードは**完成形で用意**しています。あなたの仕事は、(1) その構造を読み解いて説明すること、(2) ハイパーパラメータを変えて精度がどう動くかを実験すること、(3) 精度が伸びないときの対処を選ぶこと、です。各問のタグ：
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（層の幅・epoch・lr など）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータを変えて精度を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります（学習ループの肝など）。AI に頼り切らず、要となる処理は自分で書けることも確認します。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. 3モデルの実装
2. 学習と比較
3. 精度の棒グラフ
4. 混同行列とモデル保存

---

## この回で学ぶこと

### なぜ CNN が画像に強いのか

全結合層（MLP）の問題点：
- 28×28 = 784 ピクセルをすべてバラバラに扱う
- 「隣接するピクセルが関連している」という画像の構造的な情報を捨てる
- 画像が少し移動・回転するだけで全く異なる入力になる

CNN（Convolutional Neural Network）はこれを解決する：

```
【畳み込みの直感】
3×3 のフィルター（カーネル）が画像上をスライドしながら
局所的な特徴（エッジ，テクスチャ，形状）を検出する

第1層: エッジ検出（縦線，横線，斜め線）
第2層: テクスチャ（格子，波，点）
第3層: 形状（目，耳，数字の丸みなど）
```

### 畳み込み後の特徴マップサイズの計算

```
出力サイズ = (入力サイズ - カーネルサイズ + 2 × パディング) / ストライド + 1
```

今回の SimpleCNN：
```
入力: (28, 28)
Conv2d(1→16, kernel=3, padding=1): (28+2×1-3)/1+1 = 28 → (28, 28)
MaxPool2d(2): 28 / 2 = 14 → (14, 14)
Conv2d(16→32, kernel=3, padding=1): → (14, 14)
MaxPool2d(2): 14 / 2 = 7 → (7, 7)
→ 32チャネル × 7 × 7 = 1568次元 にFlatten
```

### MaxPooling の役割

- **ダウンサンプリング**：特徴マップを縮小して計算量を削減
- **位置不変性**：特徴が少し移動しても同じ出力（数字「3」が少し右に寄っていても認識できる）
- 各 2×2 領域の最大値を取るため，最も顕著な特徴を保持する

### モデルの保存と読み込み（state_dict）

学習済みモデルを保存する方法：
```python
torch.save(model.state_dict(), "model.pth")  # 重みパラメータのみ保存
```
モデル全体（構造 + 重み）を保存することも可能だが，`state_dict`（重みのみ）の保存が推奨される理由は：
- ファイルサイズが小さい
- Python バージョン間の互換性が高い
- 別のモデルに重みを転用しやすい（転移学習）

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 問題1　3モデルの構造を読み解く　【説明】
***

### 3モデルの役割と比較の意義

今回実装する3モデルは，複雑さの段階的な比較になっている：

| モデル | 複雑さ | 特徴 |
|---|---|---|
| `LinearNet` | 最も単純（線形） | 隠れ層なし。784→10 の直接マッピング。ベースライン |
| `DeepMLP` | 中程度（非線形，多層） | 隠れ層2つ。非線形パターンを学習できる |
| `SimpleCNN` | 最も複雑（空間構造を活用） | 畳み込みで画像の局所特徴を抽出 |

同じデータ・同じ条件で比較することで，「モデルの複雑さ」の効果を純粋に検証できる。

### `DeepMLP` の実装について

`SimpleMLP`（問題15）との違いは隠れ層が2つになること：
```
784 → [Linear(784,256)] → [ReLU] → [Linear(256,128)] → [ReLU] → [Linear(128,10)]
```

層を増やすと：
- より複雑なパターンを学習できる
- 計算量が増える
- 過学習しやすくなる（Dropoutなどの正則化が必要になることも）

### `SimpleCNN` の実装について

CNN では `nn.Sequential` を使うと層をまとめて書ける：
```python
self.conv_block = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
)
```

### 課題

下のコードセルには、3つのモデル（`LinearNet` / `DeepMLP` / `SimpleCNN`）が **完成形で実装**されています。今回はネットワークを一から書くのではなく、**「なぜこの構造でうまくいくのか」を読み解く**のが目的です。

**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたら実行し、各モデルのパラメータ数を確認してください。

説明を書くときは、次の問いを意識してください：

- `LinearNet` と `DeepMLP` の違いは何か？ なぜ MLP の方が複雑なパターンを学習できるのか？
- CNN の特徴マップのサイズはなぜ `28 → 14 → 7` と変わるのか？（`MaxPool2d(2)` の役割）
- 最後の全結合層の入力がなぜ `32 × 7 × 7` なのか？
- `kernel_size=3, padding=1` にすると畳み込み後もサイズが 28 のまま保たれるのはなぜか？

> **考察（パラメータ数）**: 出力されるパラメータ数を見て、`SimpleCNN` は `DeepMLP` より層が深いのに**パラメータ数はむしろ少ない**かもしれません。なぜだと思いますか？（ヒント：畳み込みは「同じフィルターを画像全体で使い回す」）

In [ ]:
# 完成形のモデル定義です。各行の「# 説明:」に自分の言葉で意味を書いてください。
# （コードは変更しない。説明は AI に書かせず自分で書くこと）

class LinearNet(nn.Module):
    """隠れ層なし（線形）。第7回の線形モデルに相当するベースライン。"""
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(28 * 28, 10)             # 説明:

    def forward(self, x):
        x = x.view(x.size(0), -1)                    # 説明:
        return self.fc(x)


class DeepMLP(nn.Module):
    """多層全結合 ANN。hidden1/hidden2 は実験で変えられるよう引数化。"""
    def __init__(self, hidden1=256, hidden2=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28 * 28, hidden1),             # 説明:
            nn.ReLU(),                               # 説明:（なぜ非線形が必要？）
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 10),                  # 説明:（出力が10の理由）
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


class SimpleCNN(nn.Module):
    """畳み込みで画像の局所特徴を抽出する。"""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 説明:（28x28 -> ?）
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 説明:（サイズはどう変わる？）
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 説明:
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 説明:（28->14->7 になる理由）
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 7 * 7, 128),                   # 説明:（なぜ 32*7*7 ?）
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)                         # 説明:（flatten の意味）
        return self.fc(x)


# パラメータ数の確認（実行するだけ）
for name, model in [("LinearNet", LinearNet()), ("DeepMLP", DeepMLP()), ("SimpleCNN", SimpleCNN())]:
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:10s}: パラメータ数 = {n_params:,}")


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) `LinearNet` と `DeepMLP` の違い／なぜ MLP は複雑なパターンを学習できるのか
answer_1_a = """
"""

# (1-b) CNN の特徴マップが `28 → 14 → 7` と変わる理由（`MaxPool2d(2)` の役割）
answer_1_b = """
"""

# (1-c) 最後の全結合層の入力がなぜ `32 × 7 × 7` なのか
answer_1_c = """
"""

# (1-d) 各モデルのパラメータ数（実行結果）：LinearNet=　/ DeepMLP=　/ SimpleCNN=
answer_1_d = """
"""

# (1-e) 考察：SimpleCNN は層が深いのにパラメータ数が少ない（ことがある）理由
answer_1_e = """
"""



## 問題2　学習とハイパーパラメータの実験　【骨格+実験】
***

### 「同条件」で比較することの重要性

科学的な比較には **公平な条件** が必要だ。今回は：
- エポック数：3（全モデル同じ）
- 学習率：0.001（全モデル同じ）
- バッチサイズ：64（全モデル同じ）
- データ：MNIST（全モデル同じ）

これらを統一することで，「モデルのアーキテクチャの違い」だけが結果に影響する。卒業研究でモデルを比較するときも，比較条件を統一することが必須だ。

### `train_one_model` 関数を設計する理由

同じ学習ループを3回書くのは非効率で，ミスも起きやすい。関数化することで：
- コードの重複を排除
- 引数を変えるだけで異なるモデル/設定を試せる
- バグが1箇所に集中する（修正が楽）

### 課題

下のコードセルの `train_one_model` 関数は、骨組みは用意してありますが **学習の核心（順伝播→損失→逆伝播→更新の4ステップ）はあなたが書きます**（`# ★あなたが書く★`）。ここはニューラルネット学習の心臓部なので、必ず自分で書いてください。

書けたら **同条件**（epoch=3, Adam lr=0.001）で3モデルを学習し、`results` に正解率を記録します。

次に、`EXPERIMENTS` リスト（★印）で **5通り以上**・**2つ以上の軸**（EPOCHS / LR / hidden1 など）を **1回の実行でループ**し、表示される `experiment_log_run` を **✍️ 解答用コードセルの `experiment_log`** に転記します。例：

- `EPOCHS` を 1 / 3 / 5 に変える
- `LR` を 0.01 / 0.001 / 0.0001 に変える
- `DeepMLP` の `hidden1` を 64 / 256 / 512 に変える

> **考察1**: `LinearNet < DeepMLP < SimpleCNN` の順に精度が上がりましたか？ 予想と違ったら何が原因だと思いますか？
>
> **考察2**: `LR` を大きくしすぎたとき（例 0.01 以上）に精度が不安定になることがあります。なぜ学習率が大きすぎると学習がうまくいかないのか、解答用コードセルに書いてください。

In [ ]:
# === 完成形の学習関数：中身は q15 の学習ループと同じ ===
def train_one_model(model, train_loader, test_loader, epochs=3, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            # ★あなたが書く★：学習の4ステップ（勾配初期化→順伝播で損失→逆伝播→パラメータ更新）
            #   ヒント: optimizer.zero_grad() / loss = criterion(model(images), labels)
            #           / loss.backward() / optimizer.step() の4行
            ___

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


def make_model(name):
    return {"LinearNet": LinearNet, "DeepMLP": DeepMLP, "SimpleCNN": SimpleCNN}[name]()


def build_model_for_exp(cfg):
    if cfg["model"] == "DeepMLP":
        return DeepMLP(hidden1=cfg.get("hidden1") or 256)
    return make_model(cfg["model"])


# === ★ここを変えて実験する★：設定の一覧 ===
EXPERIMENTS = [
    {"tag": "lin_base", "model": "LinearNet", "EPOCHS": 3, "LR": 0.001, "hidden1": None},
    {"tag": "mlp_base", "model": "DeepMLP", "EPOCHS": 3, "LR": 0.001, "hidden1": 256},
    {"tag": "cnn_base", "model": "SimpleCNN", "EPOCHS": 3, "LR": 0.001, "hidden1": None},
    {"tag": "mlp_ep5", "model": "DeepMLP", "EPOCHS": 5, "LR": 0.001, "hidden1": 256},
    {"tag": "mlp_lr01", "model": "DeepMLP", "EPOCHS": 3, "LR": 0.01, "hidden1": 256},
]

results = {}
experiment_log_run = []
for i, cfg in enumerate(EXPERIMENTS, start=1):
    model = build_model_for_exp(cfg)
    acc = train_one_model(
        model, train_loader, test_loader, epochs=cfg["EPOCHS"], lr=cfg["LR"]
    )
    if cfg["tag"] in ("lin_base", "mlp_base", "cnn_base"):
        results[cfg["model"]] = acc
    h1 = cfg["hidden1"] if cfg["hidden1"] is not None else "-"
    experiment_log_run.append({
        "row": i,
        "model": cfg["model"],
        "EPOCHS": cfg["EPOCHS"],
        "LR": cfg["LR"],
        "hidden1": h1,
        "test_accuracy": round(acc, 4),
    })
    print(
        f"{cfg['tag']:10s} model={cfg['model']:10s} EPOCHS={cfg['EPOCHS']} LR={cfg['LR']:<6} -> test={acc:.4f}"
    )

print("\n--- experiment_log_run（解答欄の experiment_log に転記）---")
import pandas as pd
print(pd.DataFrame(experiment_log_run))

# 問題4 の再学習用（ベースライン設定）
EPOCHS = 3
LR = 0.001


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 実験ログ（上のセルの experiment_log_run を転記。5通り以上・2軸以上）
experiment_log = pd.DataFrame([
    {'row': 1, 'model': 'LinearNet', 'EPOCHS': 3, 'LR': 0.001, 'hidden1': '-', 'test_accuracy': None},
    {'row': 2, 'model': 'DeepMLP', 'EPOCHS': 3, 'LR': 0.001, 'hidden1': 256, 'test_accuracy': None},
    {'row': 3, 'model': 'SimpleCNN', 'EPOCHS': 3, 'LR': 0.001, 'hidden1': '-', 'test_accuracy': None},
    {'row': 4, 'model': 'DeepMLP', 'EPOCHS': 5, 'LR': 0.001, 'hidden1': 256, 'test_accuracy': None},
    {'row': 5, 'model': 'DeepMLP', 'EPOCHS': 3, 'LR': 0.01, 'hidden1': 256, 'test_accuracy': None},
    {'row': 6, 'model': None, 'EPOCHS': None, 'LR': None, 'hidden1': None, 'test_accuracy': None},
])

# (2-b) 動かした「軸」（例：EPOCHS と LR）
answer_2_b = """
"""

# (2-c) 考察1：複雑なモデルほど精度が上がったか／違ったら理由
reflection1 = """
"""

# (2-d) 考察2：学習率が大きすぎると学習がうまくいかない理由
reflection2 = """
"""



## 問題3　精度比較と「伸びないときの対処」の選択　【骨格+選択】
***

### 棒グラフで比較する際のポイント

精度の差が小さい（例：97% vs 99%）場合，y軸を 0.95〜1.0 に絞ると差が見やすくなる：

```python
plt.ylim(0.95, 1.0)
```

ただし **y軸を切り捨てる（truncated y-axis）** のは，差を誇張して見せることになるため，学術的な文脈では注意が必要だ。常に「実際の差の大きさ」を意識すること。

### 課題

下のコードセルは棒グラフの体裁を用意してありますが、**核心（精度リストの取り出し）はあなたが書きます**（`# ★あなたが書く★`）。書いて実行し、結果を確認してください。

そのうえで、次の **設計判断** に答えてください。

> **設計判断**: あなたのモデルの精度が「思ったより伸びない」とき、次のうち**効果がありそうな対処を2つ以上選び**、それぞれ「なぜ効くと思うか」を解答用コードセルに書いてください。**やみくもに全部やるのではなく、状況に応じて選ぶ**のが大事です。
>
> - **(A) epoch を増やす** … 学習回数を増やす
> - **(B) 学習率(lr)を調整する** … 大きすぎ/小さすぎを直す
> - **(C) Dropout などの正則化を入れる** … 過学習を抑える
> - **(D) データ拡張（回転・平行移動）を行う** … 学習データを水増し
> - **(E) 層やチャネルを増やしてモデルを複雑にする** … 表現力を上げる
> - **(F) Batch Normalization を入れる** … 各層の出力を正規化して学習を安定・高速化
>
> ヒント：「訓練精度は高いがテスト精度が低い（＝過学習）」のか、「訓練精度自体が低い（＝表現力/学習不足）」のかで、選ぶべき対処は変わります。

> **考察3**: モデルの複雑さ（パラメータ数）と精度の関係はどうでしたか？ パラメータが多いほど常に精度が上がりましたか？


In [ ]:
# === 用意済み：モデル名の一覧 ===
names = list(results.keys())

# ★あなたが書く★：results から各モデルの正解率を取り出して accs（リスト）を作る（1行）
#   ヒント: results は {モデル名: 正解率} の辞書。names の順に値を並べる
accs = ___

# === 用意済み：棒グラフ描画 ===
plt.bar(names, accs)
plt.ylim(0.9, 1.0)  # 差を見やすくするため軸を絞っている（誇張に注意）
for i, a in enumerate(accs):
    plt.text(i, a, f"{a * 100:.2f}%", ha="center", va="bottom")
plt.ylabel("test accuracy")
plt.title("3モデルの精度比較")
plt.show()

best = max(results, key=results.get)
print("最高精度モデル:", best, f"({results[best] * 100:.2f}%)")


In [ ]:
# === ✍️ 問題3 解答（採点対象）===

# (3-a) 設計判断：精度が伸びないときに選んだ対処（A〜F から2つ以上）：(　)(　)
# (A) epoch を増やす
# (B) 学習率(lr)を調整する
# (C) Dropout などの正則化を入れる
# (D) データ拡張（回転・平行移動）を行う
# (E) 層やチャネルを増やしてモデルを複雑にする
# (F) Batch Normalization を入れる
design3_choice = ""

# (3-b) 選んだ対処1が効く理由
answer_3_b = """
"""

# (3-c) 選んだ対処2が効く理由
answer_3_c = """
"""

# (3-d) 考察3：パラメータ数と精度の関係（多いほど常に上がるか）
reflection3 = """
"""



## 問題4　混同行列の解釈とモデル保存　【説明+選択】
***

### 混同行列から何を読み取るか

10クラス（数字0〜9）の混同行列は 10×10 のヒートマップになる。対角線が正解，対角線以外が誤分類だ。

よく混同される数字の例：
- **1 と 7**: 縦棒の形が似ている
- **3 と 8**: 右側のカーブが似ている（特に崩した字体）
- **4 と 9**: 縦線と右下のカーブが似ている
- **5 と 6**: どちらも上が開いた形

「どのクラスが最も多く誤分類されているか」を確認し，その数字の特徴を考えてみよう。

### 混同行列から改善策を考える

特定のクラスで誤分類が多い場合の対処法：
- そのクラスのデータを増やす（Data Augmentation）
- クラス不均衡対策（第12回の `class_weight`）
- モデルを複雑にする
- 前処理の改善（画像の回転・スケール正規化）

### モデルの保存

第17回では保存した `best_mnist_model.pth` を読み込んで使用する。**ファイル名と保存先を正確に一致させること**が重要だ。

### 課題

下のコードセルは最高精度モデルの再学習・ヒートマップ描画・保存までを用意してありますが、**核心（テストデータの予測ラベルを集める1行）はあなたが書きます**（`# ★あなたが書く★`）。書いて実行すると、混同行列が描かれ `best_mnist_model.pth` が保存されます（第17回で使用）。

混同行列を**読み取って**、次に答えてください。

> **設計判断（読み取り＋対処）**:
> 1. あなたの混同行列で「**最も混同されている数字のペア**」を2〜3個挙げてください（解答用コードセルに記入）。
> 2. その誤分類を減らすには、問題3の対処 (A)〜(F) のうちどれが有効そうですか？ **1つ選んで理由**を書いてください。
>
> ヒント：よく混同される例 — 1と7（縦棒）/ 3と8（右のカーブ）/ 4と9（縦線と右下）/ 5と6（上が開いた形）。

> **考察4**: 混同行列の「対角線」は何を表しますか？ また、特定の数字だけ誤分類が多い場合、それはデータの問題（その数字の枚数が少ない等）かモデルの問題か、どう切り分けますか？

In [ ]:
# === 完成済みコード：そのまま実行してください ===
# 最高精度モデル（問題3の best）を再学習する
best_model = make_model(best)
train_one_model(best_model, train_loader, test_loader, epochs=EPOCHS, lr=LR)

# テストデータ全体の予測を集める
best_model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        # ★あなたが書く★：best_model の出力から予測ラベル preds を求める（1行）
        #   ヒント: model(images) の出力（logits）に .argmax(dim=1) を取り、.cpu() で CPU へ
        preds = ___
        y_pred.extend(preds.numpy())
        y_true.extend(labels.numpy())

# 混同行列のヒートマップ
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("予測")
plt.ylabel("正解")
plt.title(f"混同行列（{best}）")
plt.show()

# 第17回で使うモデルを保存
torch.save(best_model.state_dict(), "best_mnist_model.pth")
print("保存しました: best_mnist_model.pth")


In [ ]:
# === ✍️ 問題4 解答（採点対象）===

# (4-a) 混同されやすい数字のペア（2〜3個）
answer_4_a = """
"""

# (4-b) それを減らすのに有効な対処（A〜F から1つ）：(　)
# (A) データ拡張
# (B) モデルを大きくする
# (C) 学習率を調整
# (D) 混同行列を見て誤りパターンを分析
# (E) アンサンブル
# (F) 前処理を見直す
answer_4_b = ""

# (4-c) その理由
answer_4_c = """
"""

# (4-d) 考察4：混同行列の「対角線」は何を表すか
answer_4_d = """
"""

# (4-e) 考察4：特定の数字だけ誤分類が多いとき、データの問題かモデルの問題かをどう切り分けるか
answer_4_e = """
"""

